In [17]:
import sys

import numpy as np
import pandas as pd
sys.path.append("/cs/casmip/alina.ryabtsev/FewShotLearning/Lemonade")
from Lemonade.VirtualRichard import VirtualRichard
from glob import glob
from Lemonade.constants import *
import os
import nibabel as nib
from Lemonade.utils import postprocess_predictions
from Lemonade import utils
import re

In [2]:
%reload_ext autoreload
%autoreload 2

In [3]:
# Path to the folder with the scans and the corresponding predictions
predictions_files = sorted(glob(os.path.join(LIVER_LESIONS_DATASET, "*support_analysis_10.nii.gz")))
pattern = re.compile(r".*\d+_support_analysis_10.nii.gz")
predictions_files = [path for path in predictions_files if pattern.search(path)]
scans = [path.replace("_support_analysis_10.nii.gz", "_scan.nii.gz") for path in predictions_files]
liver_masks = [path.replace("_support_analysis_10.nii.gz", "_liver.nii.gz") for path in predictions_files]

In [4]:
# Load the scans and the predictions
scans = [nib.load(scan).get_fdata() for scan in scans]
predictions = [nib.load(prediction).get_fdata() for prediction in predictions_files]

In [5]:
liver_masks = [nib.load(liver_mask).get_fdata().astype("float64") for liver_mask in liver_masks]

In [6]:
# Get only predictions within the liver
predictions = [prediction * liver_mask for prediction, liver_mask in zip(predictions, liver_masks)]
# postprocess the predictions
post_predictions = postprocess_predictions(predictions, save_postprocessed=True, predictions_affines=[nib.load(path).affine for path in predictions_files], predictions_filenames=predictions_files)

100%|██████████| 93/93 [01:56<00:00,  1.26s/it]


In [7]:
# Get the voxel volume
voxel_vols = [np.prod(nib.load(path).header.get_zooms()) for path in predictions_files]

In [8]:
# Get tumor that are bigger than 10 mm in diameter
post_predictions_big = [utils.mask_by_diameter(prediction, voxel_vol, 10)[1] for prediction, voxel_vol in zip(post_predictions, voxel_vols)]

In [18]:
# Initialize the VirtualRichard
virtual_richard = VirtualRichard()

In [10]:
# get the GT masks
gt_masks = [nib.load(path.replace("_support_analysis_10.nii.gz", "_seg.nii.gz")).get_fdata() for path in predictions_files]
post_gt_masks = postprocess_predictions(gt_masks)

100%|██████████| 93/93 [01:23<00:00,  1.11it/s]


In [11]:
# Get only the tumors that are bigger than 10 mm in diameter in the GT masks
gt_masks_big = [utils.mask_by_diameter(gt_mask, voxel_vol, 10)[1] for gt_mask, voxel_vol in zip(gt_masks, voxel_vols)]
post_gt_masks_big = postprocess_predictions(gt_masks_big)

100%|██████████| 93/93 [00:47<00:00,  1.96it/s]


In [12]:
detection_metrics = virtual_richard.evaluate_detection(post_predictions, gt_masks)

In [13]:
detection_metrics_big = virtual_richard.evaluate_detection(post_predictions_big, gt_masks_big)

In [14]:
segmentation_metrics = virtual_richard.evaluate_segmentation(post_predictions, gt_masks)

In [31]:
segmentation_metrics_big = virtual_richard.evaluate_segmentation(post_predictions_big, gt_masks_big)

#### Detection metrics

In [20]:
TP_score = pd.DataFrame(detection_metrics[0])
FP_score = pd.DataFrame(detection_metrics[1])
FN_score = pd.DataFrame(detection_metrics[2])

In [21]:
TP_score.describe()

,0
count,93.000000
mean,0.410272
std,0.234219
min,0.000000
25%,0.250000
50%,0.400000
75%,0.533333
max,1.000000


In [22]:
FP_score.describe()

,0
count,93.000000
mean,0.850384
std,0.138451
min,0.363636
25%,0.809524
50%,0.900000
75%,0.947368
max,1.000000


In [23]:
FN_score.describe()

,0
count,93.000000
mean,0.589728
std,0.234219
min,0.000000
25%,0.466667
50%,0.600000
75%,0.750000
max,1.000000


In [24]:
TP_score_big = pd.DataFrame(detection_metrics_big[0])
FP_score_big = pd.DataFrame(detection_metrics_big[1])
FN_score_big = pd.DataFrame(detection_metrics_big[2])

In [25]:
TP_score_big.describe()

,0
count,93.000000
mean,0.544733
std,0.318647
min,0.000000
25%,0.333333
50%,0.500000
75%,0.750000
max,1.000000


In [26]:
FP_score_big.describe()

,0
count,93.000000
mean,0.774445
std,0.205597
min,0.166667
25%,0.642857
50%,0.812500
75%,0.970588
max,1.000000


In [27]:
FN_score_big.describe()

,0
count,93.000000
mean,0.455267
std,0.318647
min,0.000000
25%,0.250000
50%,0.500000
75%,0.666667
max,1.000000


#### Segmentation metrics

In [28]:
countour_score = pd.DataFrame(np.concatenate(segmentation_metrics[0]))
rvd_score = pd.DataFrame(segmentation_metrics[1])

In [36]:
countour_score_big = pd.DataFrame(np.concatenate(segmentation_metrics_big))

In [37]:
countour_score.describe()

,0
count,829.000000
mean,0.883581
std,0.251038
min,0.002974
25%,0.967320
50%,1.000000
75%,1.000000
max,1.000000


In [38]:
countour_score_big.describe()

,0
count,437.000000
mean,0.857901
std,0.259647
min,0.002974
25%,0.834728
50%,1.000000
75%,1.000000
max,1.000000
